In [1]:
pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 24.4 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 36.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sentence-transformers]ence-transformers]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


i searched for "embedded based labeling sentence transformers" and came across the all-MiniLM-L6-v2 Sentence Transformer, so lets have a look at that


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')


In [4]:
dept_labels = [
    "Marketing",
    "Sales",
    "Project Management",
    "IT",
    "Finance",
    "HR",
    "Other"
]

In [7]:
dept_label_embeddings = model.encode(dept_labels)

In [8]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []
for cv in cvs:
    for job in cv:
        if job["status"] == "ACTIVE":
            jobs.append(job)

df_active = pd.DataFrame(jobs)
df_active.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management


In [9]:
job_embeddings = model.encode(df_active["position"].astype(str).tolist())

In [10]:
similarity_matrix = cosine_similarity(job_embeddings, dept_label_embeddings)

In [11]:
best_label_idx = similarity_matrix.argmax(axis=1)
df_active["department_pred_embedding"] = [dept_labels[i] for i in best_label_idx]

In [ ]:
df_active[["position", "department", "department_pred_embedding"]].head(10)

,position,department,department_pred_embedding
0,Prokurist,Other,HR
1,CFO,Other,Finance
2,Betriebswirtin,Other,IT
3,Prokuristin,Other,HR
4,CFO,Other,Finance
5,Solutions Architect,Information Technology,Project Management
6,Medizintechnik Beratung,Consulting,Marketing
7,Director expansión de negocio.,Business Development,Project Management
8,Gerente comercial,Sales,Marketing
9,Administrador Unico,Administrative,HR


In [13]:
embedding_accuracy = (
    df_active["department"] == df_active["department_pred_embedding"]
).mean()

embedding_accuracy

np.float64(0.11717495987158909)

In [14]:
df_active[
    df_active["department"] != df_active["department_pred_embedding"]
][["position", "department", "department_pred_embedding"]].head(20)

,position,department,department_pred_embedding
0,Prokurist,Other,HR
1,CFO,Other,Finance
2,Betriebswirtin,Other,IT
3,Prokuristin,Other,HR
4,CFO,Other,Finance
5,Solutions Architect,Information Technology,Project Management
6,Medizintechnik Beratung,Consulting,Marketing
7,Director expansión de negocio.,Business Development,Project Management
8,Gerente comercial,Sales,Marketing
9,Administrador Unico,Administrative,HR


So we can see that the department Label "Other" dominates. This is the main couse of the low accuracy. Maybe we should get the Label Other correct.

As an optional extension, we take the information from previous positions to provide additional context for active roles labeled as "Other", because its unlikely to change the Department.

Lets only look at the historical jobs which have no Label "Other" and which are INACTIVE 

In [ ]:
from collections import Counter
len(cvs)

def dept_from_history(cvs):
    past_departments = [
        job["department"]
        for job in cvs
        if job["status"] == "INACTIVE" and job["department"] != "Other"
    ]

    if not past_departments:   # If there are no historical Jobs with clear Department return None
        return None
    most_common, count = Counter(past_departments).most_common(1)[0]
    return most_common

In [ ]:
rows = []

for cv in cvs:  # cv = eine Person
    
    active_jobs = [
        job for job in cv
        if job["status"] == "ACTIVE"
    ]
    
    if not active_jobs:
        continue  
    
    active_job = active_jobs[0]  
    
    
    if active_job["department"] != "Other":
        continue
    
    inferred_dept = dept_from_history(cv)
    
    rows.append({
        "position": active_job["position"],
        "original_department": active_job["department"],
        "inferred_department_from_history": inferred_dept
    })

In [24]:
df_other_extension["inferred_department_from_history"].notna().mean()

np.float64(0.496)

In [25]:
active_counts = [sum(1 for job in cv if job["status"] == "ACTIVE") for cv in cvs]
pd.Series(active_counts).value_counts().head(10)

1     380
0     131
2      78
3      10
5       4
4       3
8       1
7       1
10      1
Name: count, dtype: int64